> ⏸ **Parked: not part of the current phase.** This notebook will be updated once model training and evaluation (notebooks 01–03) are finalised. Don't run it yet.

# 00 · All-in-one pipeline (FaceForensics++)

Runs every phase in one session. For the step-by-step version with richer analysis, use notebooks 01–07.

**Before running, set these in the notebook sidebar:**
1. **Accelerator:** GPU **T4 x2** (not P100: recent PyTorch builds no longer support the P100).
2. **Internet:** On (needed for pip, the face detector, pretrained EfficientNet weights and the Claude API).
3. **Add Input → Datasets:** a FaceForensics++ **video** dataset (e.g. a c23 mirror with `original/`, `Deepfakes/`, `Face2Face/`, `FaceSwap/`, `NeuralTextures/` folders, or the official `original_sequences/` + `manipulated_sequences/` layout).
4. **Project code:** either add a Kaggle dataset made from a zip of this repo, or clone it from GitHub (cell 1).
5. *(Optional, for the explanation evaluation)*: **Add-ons → Secrets** → add your LLM key (`ANTHROPIC_API_KEY`, `OPENROUTER_API_KEY`, `OPENAI_API_KEY` or `LLM_API_KEY`), or set `LLM_PROVIDER = "ollama"` to run a local model.

In [ ]:
# --- Project code -------------------------------------------------------------------------
# Uses the repo if it is attached as an input (zip uploaded as a Kaggle dataset, or an earlier
# notebook's output); otherwise clones it. For a private repo add a GITHUB_TOKEN secret.
import os, sys, glob, shutil, subprocess

PROJECT = "/kaggle/working/project"
REPO, BRANCH = "https://github.com/Amay-M-Nair/DFDC.git", "dev"

if not os.path.exists(PROJECT):
    found = [p for d in range(5) for p in glob.glob("/kaggle/input/" + "*/" * d + "src/config.py")]
    if found:
        shutil.copytree(os.path.dirname(os.path.dirname(found[0])), PROJECT,
                        ignore=shutil.ignore_patterns("data", "outputs", "__pycache__"))
    else:
        token = ""
        try:
            from kaggle_secrets import UserSecretsClient
            token = UserSecretsClient().get_secret("GITHUB_TOKEN")
        except Exception:
            pass
        url = REPO.replace("https://", f"https://{token}@") if token else REPO
        r = subprocess.run(["git", "clone", "-q", "-b", BRANCH, url, PROJECT], capture_output=True)
        if r.returncode:
            raise RuntimeError("git clone failed: attach the project as a dataset or add a GITHUB_TOKEN secret")
        subprocess.run(["git", "-C", PROJECT, "remote", "set-url", "origin", REPO])  # don't keep the token

os.chdir(PROJECT)
sys.path.insert(0, PROJECT)
!pip install -q grad-cam anthropic
print("project ready:", sorted(os.listdir(PROJECT)))

In [ ]:
# timm, torch, sklearn, opencv are preinstalled on Kaggle; these are not
!pip install -q grad-cam anthropic

import torch
print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
# Find the FaceForensics++ root (folder containing the videos) in /kaggle/input
vids = glob.glob("/kaggle/input/**/*.mp4", recursive=True)
assert vids, "Add a FaceForensics++ video dataset as input"
FF_SRC = os.path.commonpath([os.path.dirname(v) for v in vids])
print("FF++ root:", FF_SRC, "|", len(vids), "videos")

# Face crops are written to /kaggle/working (the input is read-only)
os.environ["DFD_DATA_DIR"] = "/kaggle/working/data"
os.environ["DFD_OUTPUT_DIR"] = "/kaggle/working/outputs"

## Phase 0: extract face crops from the videos
Splits are made by identity (no face appears in two splits). `--max-videos 400` takes about 15–25 min; remove it to use every video (about 1 hour). If the dataset ships the official split files, add `--splits <dir with train.json/val.json/test.json>`.

In [ ]:
!python -m src.classifier.ff_extract --src "{FF_SRC}" --frames 10 --max-videos 400 --workers 4
!du -sh /kaggle/working/data

## Phase 1: train EfficientNet-B0

In [ ]:
!python -m src.classifier.train --epochs 8 --workers 4

In [ ]:
!python -m src.classifier.evaluate --workers 4
import json
from IPython.display import Image as Show, display
m = json.load(open("/kaggle/working/outputs/metrics/test_metrics.json"))
print(json.dumps({k: m[k] for k in ("accuracy", "roc_auc", "video_level", "per_method")}, indent=2))
display(Show("/kaggle/working/outputs/metrics/test_roc.png"), Show("/kaggle/working/outputs/metrics/test_confusion.png"))

## Phase 2: Grad-CAM++

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
from src.xai.gradcam import Explainer, draw_bbox

cam = Explainer()
samples = sorted(glob.glob(os.environ["DFD_DATA_DIR"] + "/test/fake/*"))[:3] + sorted(glob.glob(os.environ["DFD_DATA_DIR"] + "/test/real/*"))[:3]
fig, axes = plt.subplots(2, 6, figsize=(20, 7))
for i, path in enumerate(samples):
    img = Image.open(path)
    exp = cam(img)
    axes[0, i].imshow(img); axes[0, i].set_title(path.split("/")[-2]); axes[0, i].axis("off")
    axes[1, i].imshow(draw_bbox(exp.overlay, exp.bbox)); axes[1, i].set_title(f"{exp.label} {exp.prob_fake:.2f}
{exp.top_regions[0][0]}", fontsize=9); axes[1, i].axis("off")
plt.tight_layout(); plt.show()
print(exp.describe())

## Phases 3–5: ungrounded vs. RAG-grounded explanations

In [ ]:
from src.kaggle_utils import setup_llm, start_ollama

LLM_PROVIDER = "anthropic"   # anthropic | openrouter | ollama | openai (any OpenAI-compatible server)
LLM_MODEL = None             # None = provider default
if LLM_PROVIDER == "ollama":
    start_ollama(LLM_MODEL or "llama3.1:8b")
LLM_READY = setup_llm(LLM_PROVIDER, LLM_MODEL)

In [ ]:
import json
import pandas as pd

if LLM_READY:
    !python -m src.eval.run_eval --n 50
    summary = json.load(open("/kaggle/working/outputs/eval/summary.json"))
    print("LLM:", summary.pop("llm"))
    display(pd.DataFrame(summary))

## Optional: demo UI
Kaggle can't open localhost, so `--share` creates a temporary public gradio.live link. Stop the cell to shut it down.

In [ ]:
# !pip install -q gradio && python app.py --share